In [45]:
import pandas as pd
import unicodedata

def remove_accents(text):
    return "".join(
        c for c in unicodedata.normalize("NFKD", str(text))
        if not unicodedata.combining(c)
    )

In [53]:
# get LA sweet spot data stats from csv
launch_angle_data = pd.read_csv('launch_angle_data/savant_launch_angle_data.csv')
launch_angle_data["Name"] = launch_angle_data["last_name, first_name"].str.split(", ").str[::-1].str.join(" ")

launch_angle_data["Name"] = launch_angle_data["Name"].apply(remove_accents)

launch_angle_data = launch_angle_data[['Name', 'Season','sweet_spot_percent']]

In [54]:
# get team player PA data from csv
team_player_pa = pd.read_csv('launch_angle_data/player_team_pa.csv')

In [55]:
# get team wrc+ from fangraphs api

START = 2015
END = 2025

all_years = []

for year in range(START, END + 1):

    url = (
        f"https://www.fangraphs.com/api/leaders/major-league/data"
        f"?pos=all"
        f"&stats=bat"
        f"&lg=all"
        f"&qual=0"
        f"&season={year}"
        f"&season1={year}"
        f"&team=0,ts"
        f"&ind=0"
        f"&type=8"
        f"&pageitems=1000"
    )

    df = pd.read_json(url)

    rows = []

    for i in range(len(df)):

        team = df.iloc[i]["data"]   # one team dict

        rows.append({
            "Season": year,
            "Team": team["TeamNameAbb"],
            "wRC+": team["wRC+"]
        })

    all_years.append(pd.DataFrame(rows))

team_wrc = pd.concat(
    all_years,
    ignore_index=True
)

In [56]:
team_wrc.head()

,Season,Team,wRC+
0,2015,TOR,117.350959
1,2015,SFG,103.715278
2,2015,LAD,106.176802
3,2015,HOU,108.353481
4,2015,CHC,95.735996


In [57]:
team_player_pa.head()

,Name,Team,pos,Season,PA
0,Mike Trout,LAA,OF,2018,608
1,Mike Trout,LAA,OF,2015,682
2,Mike Trout,LAA,OF,2016,681
3,Mike Trout,LAA,OF,2019,600
4,Mike Trout,LAA,OF,2017,507


In [58]:
launch_angle_data.head()


,Name,Season,sweet_spot_percent
0,Bartolo Colon,2015,23.1
1,Torii Hunter,2015,28.5
2,David Ortiz,2015,34.8
3,Alex Rodriguez,2015,31.4
4,Aramis Ramirez,2015,33.5


In [59]:
full_data = team_player_pa.merge(launch_angle_data, on=['Name', 'Season'], how="left")
full_data_filtered = full_data.dropna()
full_data_filtered = full_data_filtered[full_data_filtered['PA'] > 25]

In [61]:
full_data_filtered.to_csv('launch_angle_data/player_data.csv')
team_wrc.to_csv('launch_angle_data/team_data.csv')